# Nacimientos diarios en EE. UU.: forecasting

El objetivo es estimar nacimientos diarios futuros usando datos de 2000 a 2014. El calendario incluye día de la semana, festivos federales y viernes 13.

**En sencillo:** los nacimientos cambian según el día de la semana y otras fechas del calendario. Como queremos predecir el futuro, entrenamos con los días antiguos y reservamos los más recientes para probar el modelo; nunca barajamos las fechas.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from ml_al_alfa.datasets import FORECAST_CONFIG, load_forecast_frame
from ml_al_alfa.forecasting import forecast_to_date, train_forecast_and_save
from ml_al_alfa.train import ARTIFACTS_DIR, RAW_DIR

sns.set_theme(style="whitegrid")
DATASET = "births"
CONFIG = FORECAST_CONFIG[DATASET]
TARGET = CONFIG["target"]

## 1. Cargar y revisar la serie

El CSV público se descarga si no existe y se guarda en `data/raw/`.

In [ ]:
series = load_forecast_frame(DATASET, RAW_DIR)
series["date"] = pd.to_datetime(series["date"])
print(f"Filas: {len(series):,}; fechas: {series['date'].min().date()} – {series['date'].max().date()}")
print(f"Fechas repetidas: {series['date'].duplicated().sum()}")
print(f"Fechas u objetivos ausentes: {series[['date', TARGET]].isna().sum().sum()}")
display(series.head())

## 2. Explorar el calendario y la serie

Observa el patrón semanal y los valores a lo largo del tiempo. La serie anual puede incluir eventos que un modelo sencillo no anticipa.

In [ ]:
series = series.sort_values("date").reset_index(drop=True)
series["día de la semana"] = series["date"].dt.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
axes[0].plot(series["date"], series[TARGET], linewidth=0.7)
axes[0].set(title="Nacimientos registrados cada día", xlabel="Fecha", ylabel="Nacimientos")
sns.boxplot(data=series, x="día de la semana", y=TARGET, order=weekday_order, ax=axes[1])
axes[1].tick_params(axis="x", rotation=25)
axes[1].set(title="Distribución de nacimientos por día de la semana", xlabel="Día", ylabel="Nacimientos")
plt.tight_layout()
plt.show()

series["es_viernes_13"] = (series["date"].dt.day == 13) & (series["date"].dt.dayofweek == 4)
display(series.groupby("es_viernes_13")[TARGET].agg(["count", "mean", "median"]).rename(index={False: "Otro día", True: "Viernes 13"}))

## 3. Entrenar y probar respetando el tiempo

El 20% más reciente es el conjunto de test. El baseline pronostica el valor observado siete días antes. También comparamos una regresión lineal y un bosque aleatorio.

In [ ]:
bundle = train_forecast_and_save(
    series[["date", TARGET]],
    dataset_name=DATASET,
    output_path=ARTIFACTS_DIR / "births_model.joblib",
)
metrics = pd.DataFrame(bundle["metrics"]).T.sort_values("mae")
display(metrics.rename(columns={"mae": "MAE (nacimientos/día, menor es mejor)", "r2": "R²"}))
print(f"Fecha de inicio del test: {bundle['forecast']['split_date']}")
print(f"Mejor método: {bundle['model_name']}")
print(f"Modelo para Streamlit: {ARTIFACTS_DIR / 'births_model.joblib'}")

## 4. Ver las predicciones del test

Este gráfico compara observaciones reales y pronósticos para una parte de las fechas reservadas. El MAE expresa el tamaño medio del error en nacimientos por día.

In [ ]:
evaluation = pd.DataFrame(bundle["forecast"]["evaluation"])
evaluation["date"] = pd.to_datetime(evaluation["dates"])
evaluation = evaluation.rename(columns={"actual": "Real", "predicted": "Pronóstico"})
display(evaluation[["date", "Real", "Pronóstico"]].tail(10))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(evaluation["date"], evaluation["Real"], label="Real", linewidth=1)
ax.plot(evaluation["date"], evaluation["Pronóstico"], label="Pronóstico", linewidth=1)
ax.set(title="Evaluación en fechas que no se usaron para entrenar", xlabel="Fecha", ylabel="Nacimientos")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Pronosticar días futuros

El ejemplo genera una semana a partir del último dato. Para horizontes de varios días, el modelo usa sus propias predicciones anteriores; por eso la incertidumbre aumenta cuanto más lejos se pronostica.

In [ ]:
future = forecast_to_date(
    bundle,
    variant_label=bundle["forecast"]["best_variant"],
    days_ahead=7,
)
future_frame = pd.DataFrame(future, columns=["date", "estimated_births"])
display(future_frame)
print("Son estimaciones estadísticas, no un conteo oficial ni una certeza sobre los nacimientos futuros.")